# M66 Generalized GA Service Optimization

 GA Optimization and Scenario Integration

This notebook takes the Genetic Algorithm from `m66_baseline_simulation.ipynb` and makes it reusable across scenarios. The GA rules, fitness objective, and final-backlog safeguard are unchanged from the Midterm. The only change is that every scenario-varying input (service budget, hourly trip bounds, reference plan, passenger streams, upstream occupancy, alighting targets, and the feasibility reference) is now passed in through a single scenario context instead of being read from fixed baseline globals.

**Provenance note.** Sections 1 to 6 are reused unmodified from the baseline notebook so this notebook rebuilds the same simulation engine and the same 30-replication baseline result. Sections 7 onward are generalized or extended from the baseline optimization code, and each generalized function is marked in a comment. Section 8 verifies that the generalized GA reproduces the Midterm GA result on Baseline before any scenario is integrated.

AI-assisted suggestions were used as a starting point for some Python code, following the same practice documented in the baseline notebook. The code was adapted to this project, run against the actual project data, and checked against the Midterm GA result before handoff.

In [1]:
# DATA SOURCES
# MTA Bus Stop Level Ridership: Beginning 2024
# https://data.ny.gov/d/fvdm-uavx
#
# MTA Developer Resources / Static GTFS
# https://www.mta.info/developers
#
# GTFS Schedule Reference
# https://gtfs.org/documentation/schedule/reference/
#
# DEVELOPMENT AND DOCUMENTATION NOTE
# AI-assisted suggestions were used as a starting point for some Python code.
# The group reviewed the simulation rules, adapted the code to the M66 project,
# ran the notebook using the actual project data, checked the outputs, and kept
# the final code that matched the project design.
#
# Official MTA, GTFS, pandas, NumPy, SciPy, Matplotlib, and Python documentation
# was used to check the functions and procedures in this notebook.

In [2]:
# Import the libraries used for data handling, simulation, statistics, and plots.
from pathlib import Path
import heapq
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import t

## 1. Load and Prepare Passenger Demand

In [3]:
# Define the main project directories and input file locations.
project_dir = Path("..")

raw_dir = project_dir / "data" / "raw"
processed_dir = project_dir / "data" / "processed"
gtfs_dir = raw_dir / "gtfs_m"

figures_dir = project_dir / "output" / "figures"
tables_dir = project_dir / "output" / "tables"

clean_file = processed_dir / "MTA_M66_Selected_Stops_2025_CLEAN.csv"
raw_file = raw_dir / "MTA_M66_Eastbound_2025_RAW.csv"

processed_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)
tables_dir.mkdir(parents=True, exist_ok=True)

# Check that all required simulation input files are available before continuing.
required_inputs = [
    clean_file,
    raw_file,
    gtfs_dir / "trips.txt",
    gtfs_dir / "stop_times.txt",
    gtfs_dir / "feed_info.txt",
]

missing_inputs = [path for path in required_inputs if not path.exists()]

if missing_inputs:
    missing_list = "\n".join(f"- {path}" for path in missing_inputs)
    raise FileNotFoundError(
        "Required simulation input files are missing:\n"
        f"{missing_list}"
    )

In [4]:
# Current Midterm simulation and optimization configuration.

bus_capacity = 60

morning_hours = [6, 7, 8, 9, 10]
evening_hours = [15, 16, 17, 18, 19]
study_hours = morning_hours + evening_hours

selected_stops = [6, 7, 8, 9, 10]

base_replications = 30
master_seed = 2026

# The AI screening uses the first 5 matched replications during search.
optimization_replications = list(range(1, 6))

# Seeds and search budget used for the completed GA, SA, and PSO screening.
optimizer_seeds = [101, 202, 303]
evaluation_budget = 10

In [5]:
# Reference: pandas.read_csv()
# https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html

# Load the cleaned five-stop ridership dataset.
df = pd.read_csv(clean_file)

# Convert Date and Hour into formats used by the simulation.
df["Date"] = pd.to_datetime(df["Date"])
df["Hour"] = pd.to_datetime(df["Hour"]).dt.hour

print("Cleaned rows:", len(df))
print("Study hours:", sorted(df["Hour"].unique()))
print("Selected stops:", sorted(df["Stop Sequence"].unique()))

df.head()

Cleaned rows: 12814
Study hours: [np.int32(6), np.int32(7), np.int32(8), np.int32(9), np.int32(10), np.int32(15), np.int32(16), np.int32(17), np.int32(18), np.int32(19)]
Selected stops: [np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10)]


,Date,Hour,Route ID,Direction,Stop ID,Stop Sequence,Boardings,Alightings,Trips,Stop Name,Period
0,2025-01-01,9,M66,E,403497,10,2,1,1,E 68 ST/LEXINGTON AV,Morning
1,2025-01-01,15,M66,E,403492,6,8,0,1,W 65 ST/CENTRAL PARK WEST,Evening
2,2025-01-01,15,M66,E,404994,8,0,4,1,MADISON AV/E 65 ST,Evening
3,2025-01-01,15,M66,E,403495,9,3,2,2,MADISON AV/E 67 ST,Evening
4,2025-01-01,15,M66,E,403497,10,0,1,1,E 68 ST/LEXINGTON AV,Evening


In [6]:
# Reference: pandas DataFrame.groupby()
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html

# Calculate the representative mean Boardings and Alightings for each stop-hour.
demand_table = (
    df.groupby(["Stop Sequence", "Stop Name", "Hour"])[["Boardings", "Alightings"]]
    .mean()
    .reset_index()
    .sort_values(["Stop Sequence", "Hour"])
    .reset_index(drop=True)
)

# Project Deliverable 1 defines deterministic rounding for whole passengers:
# max(0, floor(p + 0.5)).
#
# Reference: NumPy floor()
# https://numpy.org/doc/stable/reference/generated/numpy.floor.html

baseline_demand = demand_table[
    ["Stop Sequence", "Stop Name", "Hour", "Boardings"]
].copy()

baseline_demand["Passenger Count"] = np.maximum(
    0,
    np.floor(baseline_demand["Boardings"] + 0.5)
).astype(int)

print("Passenger total per replication:", baseline_demand["Passenger Count"].sum())
baseline_demand.head(10)

Passenger total per replication: 776


,Stop Sequence,Stop Name,Hour,Boardings,Passenger Count
0,6,W 65 ST/CENTRAL PARK WEST,6,7.543307,8
1,6,W 65 ST/CENTRAL PARK WEST,7,25.386100,25
2,6,W 65 ST/CENTRAL PARK WEST,8,40.393822,40
3,6,W 65 ST/CENTRAL PARK WEST,9,24.162791,24
4,6,W 65 ST/CENTRAL PARK WEST,10,18.968750,19
5,6,W 65 ST/CENTRAL PARK WEST,15,33.683398,34
6,6,W 65 ST/CENTRAL PARK WEST,16,28.030888,28
7,6,W 65 ST/CENTRAL PARK WEST,17,31.360153,31
8,6,W 65 ST/CENTRAL PARK WEST,18,22.000000,22
9,6,W 65 ST/CENTRAL PARK WEST,19,11.316206,11


In [7]:
# Individual passenger arrival times are sampled within each hour because the
# MTA ridership file contains hourly totals instead of passenger timestamps.
#
# Kieu and Cai (2018) discuss stochastic passenger-arrival modeling for public
# transport stops. For this project, the hourly passenger count stays fixed and
# only the within-hour arrival times change between replications.
# https://doi.org/10.1049/iet-its.2018.0085
#
# Reference: NumPy Generator.uniform()
# https://numpy.org/doc/stable/reference/random/generated/numpy.random.Generator.uniform.html

def generate_baseline_passengers(seed, replication):
    rng = np.random.default_rng(seed)
    records = []

    for _, row in baseline_demand.iterrows():
        start_minute = int(row["Hour"]) * 60
        end_minute = start_minute + 60

        arrival_times = rng.uniform(
            start_minute,
            end_minute,
            int(row["Passenger Count"])
        )

        for arrival_time in sorted(arrival_times):
            records.append({
                "Stop Sequence": int(row["Stop Sequence"]),
                "Stop Name": row["Stop Name"],
                "Hour": int(row["Hour"]),
                "Period": "Morning" if row["Hour"] < 12 else "Evening",
                "Arrival Minutes": float(arrival_time)
            })

    passengers = pd.DataFrame(records)

    passengers.insert(
        0,
        "Passenger ID",
        [
            f"R{replication:02d}_P{i:05d}"
            for i in range(1, len(passengers) + 1)
        ]
    )

    passengers["Replication"] = int(replication)
    passengers["Seed"] = int(seed)

    return passengers

## 2. Load and Prepare the Baseline GTFS Service

In [8]:
# Load the current MTA GTFS files.
trips = pd.read_csv(gtfs_dir / "trips.txt")
stop_times = pd.read_csv(gtfs_dir / "stop_times.txt")
feed_info = pd.read_csv(gtfs_dir / "feed_info.txt")

# Keep the weekday eastbound M66 full-route pattern.
m66_trips = trips[
    (trips["route_id"] == "M66")
    & (trips["service_id"] == "MQ_C6-Weekday")
    & (trips["direction_id"] == 0)
    & (trips["shape_id"] == "M660030")
].copy()

print("Full-route weekday trips:", len(m66_trips))

feed_info[
    ["feed_publisher_name", "feed_start_date", "feed_end_date", "feed_version"]
].head()

Full-route weekday trips: 138


,feed_publisher_name,feed_start_date,feed_end_date,feed_version
0,MTA New York City Transit,20260627,20260905,gtfs_m_20260611T155328Z


In [9]:
# Keep GTFS stop-time records for the five selected stops.
selected_stop_times = stop_times[
    stop_times["trip_id"].isin(m66_trips["trip_id"])
    & stop_times["stop_sequence"].between(6, 10)
].copy()

# Use Stop Sequence 6 to assign each trip to a study hour.
first_stop_times = selected_stop_times[
    selected_stop_times["stop_sequence"] == 6
].copy()

first_stop_times["Hour"] = (
    first_stop_times["arrival_time"]
    .str.split(":")
    .str[0]
    .astype(int)
)

study_trip_ids = first_stop_times[
    first_stop_times["Hour"].isin(study_hours)
]["trip_id"]

study_stop_times = selected_stop_times[
    selected_stop_times["trip_id"].isin(study_trip_ids)
].copy()

# Convert GTFS time text into minutes from midnight.
def time_to_minutes(time_text):
    hours, minutes, seconds = map(int, time_text.split(":"))
    return (hours * 60) + minutes + (seconds / 60)

study_stop_times["arrival_minutes"] = (
    study_stop_times["arrival_time"].apply(time_to_minutes)
)

# Count the baseline scheduled trips by hour.
trips_per_hour = (
    first_stop_times[
        first_stop_times["Hour"].isin(study_hours)
    ]
    .groupby("Hour")
    .size()
)

print("Study-period trips:", study_trip_ids.nunique())
print("Study-period stop events:", len(study_stop_times))
print("\nTrips per hour:")
print(trips_per_hour)

Study-period trips: 87
Study-period stop events: 435

Trips per hour:
Hour
6      5
7     12
8     12
9      8
10     6
15     8
16    12
17    11
18     8
19     5
dtype: int64


In [10]:
# Create the baseline service-plan table.
baseline_service_plan = trips_per_hour.reset_index(name="Trips")

baseline_service_plan["Period"] = np.where(
    baseline_service_plan["Hour"] < 12,
    "Morning",
    "Evening"
)

service_budget = (
    baseline_service_plan
    .groupby("Period")["Trips"]
    .sum()
    .to_dict()
)

print("Morning service budget:", service_budget["Morning"])
print("Evening service budget:", service_budget["Evening"])

baseline_service_plan

Morning service budget: 43
Evening service budget: 44


,Hour,Trips,Period
0,6,5,Morning
1,7,12,Morning
2,8,12,Morning
3,9,8,Morning
4,10,6,Morning
5,15,8,Evening
6,16,12,Evening
7,17,11,Evening
8,18,8,Evening
9,19,5,Evening


## 3. Prepare Upstream Occupancy and Alighting Inputs

In [11]:
# Load the raw eastbound M66 ridership file for upstream occupancy estimation.
raw_m66 = pd.read_csv(raw_file)

raw_m66["Date"] = pd.to_datetime(raw_m66["Date"])
raw_m66["Hour"] = pd.to_datetime(raw_m66["Hour"]).dt.hour

# Identify the five physical upstream stops from the same full-route GTFS pattern.
upstream_gtfs_stops = (
    stop_times[
        stop_times["trip_id"].isin(m66_trips["trip_id"])
        & stop_times["stop_sequence"].between(1, 5)
    ][["stop_id", "stop_sequence"]]
    .drop_duplicates()
    .sort_values("stop_sequence")
)

upstream_stop_ids = (
    upstream_gtfs_stops["stop_id"]
    .astype(int)
    .tolist()
)

# Keep weekday study-hour records from only those verified upstream stops.
upstream_df = raw_m66[
    (raw_m66["Date"].dt.dayofweek < 5)
    & (raw_m66["Stop ID"].isin(upstream_stop_ids))
    & (raw_m66["Hour"].isin(study_hours))
].copy()

# Remove rows with passenger activity but zero observed trips.
upstream_clean = upstream_df[
    ~(
        (upstream_df["Trips"] == 0)
        & (
            (upstream_df["Boardings"] > 0)
            | (upstream_df["Alightings"] > 0)
        )
    )
].copy()

print("Verified upstream Stop IDs:", upstream_stop_ids)
print("Upstream rows before cleaning:", len(upstream_df))
print("Upstream rows after cleaning:", len(upstream_clean))

Verified upstream Stop IDs: [403487, 403488, 403489, 403491, 403573]
Upstream rows before cleaning: 10870
Upstream rows after cleaning: 10867


In [12]:
# Calculate mean upstream Boardings and Alightings by physical stop and hour.
upstream_stop_hour = (
    upstream_clean
    .groupby(["Stop ID", "Hour"])[["Boardings", "Alightings"]]
    .mean()
    .reset_index()
)

# Sum the stop-level means to estimate the representative upstream passenger flow.
upstream_hourly_flow = (
    upstream_stop_hour
    .groupby("Hour")[["Boardings", "Alightings"]]
    .sum()
    .reset_index()
    .rename(
        columns={
            "Boardings": "Average Upstream Boardings",
            "Alightings": "Average Upstream Alightings"
        }
    )
)

upstream_hourly_flow["Upstream Net Flow"] = (
    upstream_hourly_flow["Average Upstream Boardings"]
    - upstream_hourly_flow["Average Upstream Alightings"]
)

upstream_hourly_flow["Scheduled Buses"] = (
    upstream_hourly_flow["Hour"].map(trips_per_hour)
)

upstream_hourly_flow["Initial Occupancy Unrounded"] = (
    upstream_hourly_flow["Upstream Net Flow"]
    / upstream_hourly_flow["Scheduled Buses"]
)

# Apply deterministic rounding and the 60-passenger capacity.
upstream_hourly_flow["Initial Occupancy"] = np.minimum(
    bus_capacity,
    np.maximum(
        0,
        np.floor(upstream_hourly_flow["Initial Occupancy Unrounded"] + 0.5)
    )
).astype(int)

upstream_hourly_flow[
    [
        "Hour",
        "Upstream Net Flow",
        "Scheduled Buses",
        "Initial Occupancy Unrounded",
        "Initial Occupancy"
    ]
]

,Hour,Upstream Net Flow,Scheduled Buses,Initial Occupancy Unrounded,Initial Occupancy
0,6,60.132476,5,12.026495,12
1,7,163.631926,12,13.635994,14
2,8,211.704836,12,17.642070,18
3,9,124.995854,8,15.624482,16
4,10,91.587024,6,15.264504,15
5,15,137.208668,8,17.151084,17
6,16,122.268987,12,10.189082,10
7,17,118.860856,11,10.805532,11
8,18,95.938313,8,11.992289,12
9,19,49.363709,5,9.872742,10


In [13]:
# Assign the hourly starting occupancy to each baseline GTFS trip.
initial_occupancy_by_hour = (
    upstream_hourly_flow
    .set_index("Hour")["Initial Occupancy"]
    .to_dict()
)

trip_initial_occupancy = first_stop_times[
    first_stop_times["Hour"].isin(study_hours)
][["trip_id", "Hour"]].copy()

trip_initial_occupancy["Initial Occupancy"] = (
    trip_initial_occupancy["Hour"].map(initial_occupancy_by_hour)
)

trip_initial_occupancy["Period"] = np.where(
    trip_initial_occupancy["Hour"] < 12,
    "Morning",
    "Evening"
)

print("Trips with starting occupancy:", len(trip_initial_occupancy))
print("Missing starting occupancies:", trip_initial_occupancy["Initial Occupancy"].isna().sum())

Trips with starting occupancy: 87
Missing starting occupancies: 0


In [14]:
# Convert the representative hourly alighting means into whole-number targets.
hourly_alighting_targets = demand_table[
    ["Stop Sequence", "Stop Name", "Hour", "Alightings"]
].copy()

hourly_alighting_targets["Alighting Target"] = np.maximum(
    0,
    np.floor(hourly_alighting_targets["Alightings"] + 0.5)
).astype(int)

# Build one record for every baseline bus-stop event.
bus_alighting_plan = study_stop_times[
    ["trip_id", "stop_sequence", "arrival_minutes"]
].copy()

bus_alighting_plan = bus_alighting_plan.merge(
    trip_initial_occupancy[["trip_id", "Hour", "Period"]],
    on="trip_id",
    how="left"
)

bus_alighting_plan = bus_alighting_plan.rename(
    columns={"stop_sequence": "Stop Sequence"}
)

bus_alighting_plan = bus_alighting_plan.merge(
    hourly_alighting_targets[
        ["Stop Sequence", "Hour", "Alighting Target"]
    ],
    on=["Stop Sequence", "Hour"],
    how="left"
)

bus_alighting_plan = bus_alighting_plan.sort_values(
    ["Stop Sequence", "Hour", "arrival_minutes"]
).reset_index(drop=True)

bus_alighting_plan["Assigned Alightings"] = 0

# Use quotient and remainder so every stop-hour target is preserved.
for (stop, hour), group in bus_alighting_plan.groupby(
    ["Stop Sequence", "Hour"]
):
    target = int(group["Alighting Target"].iloc[0])
    bus_count = len(group)

    quotient = target // bus_count
    remainder = target % bus_count

    assignments = [quotient + 1] * remainder
    assignments += [quotient] * (bus_count - remainder)

    bus_alighting_plan.loc[
        group.index,
        "Assigned Alightings"
    ] = assignments

alighting_check = (
    bus_alighting_plan
    .groupby(["Stop Sequence", "Hour"])
    .agg(
        Target=("Alighting Target", "first"),
        Assigned=("Assigned Alightings", "sum")
    )
    .reset_index()
)

print(
    "All baseline alighting targets preserved:",
    (alighting_check["Target"] == alighting_check["Assigned"]).all()
)

All baseline alighting targets preserved: True


In [15]:
# Create baseline lookup dictionaries used by the event handlers.
initial_occupancy_lookup = (
    trip_initial_occupancy
    .set_index("trip_id")["Initial Occupancy"]
    .to_dict()
)

alighting_lookup = (
    bus_alighting_plan
    .set_index(["trip_id", "Stop Sequence"])["Assigned Alightings"]
    .to_dict()
)

# Create a trip-to-period lookup.
trip_period_lookup = (
    trip_initial_occupancy
    .set_index("trip_id")["Period"]
    .to_dict()
)

print("Initial occupancy values:", len(initial_occupancy_lookup))
print("Bus-stop alighting values:", len(alighting_lookup))

Initial occupancy values: 87
Bus-stop alighting values: 435


## 4. Discrete-Event Simulation Engine

In [16]:
# Python heapq is used as the chronological event queue.
# https://docs.python.org/3/library/heapq.html
#
# If a bus and passenger have the exact same timestamp, the bus is processed
# first. The tuple priority uses bus=0 and passenger=1.

def handle_passenger_arrival(
    event_time,
    stop_sequence,
    passenger_id,
    stop_queues,
    queue_history,
    peak_queue,
    peak_queue_time
):
    # Add the passenger to the end of the FIFO queue.
    stop_queues[stop_sequence].append(
        (passenger_id, event_time)
    )

    current_queue = len(stop_queues[stop_sequence])

    queue_history.append({
        "Time": event_time,
        "Stop Sequence": stop_sequence,
        "Queue Length": current_queue,
        "Event": "passenger_arrival"
    })

    if current_queue > peak_queue[stop_sequence]:
        peak_queue[stop_sequence] = current_queue
        peak_queue_time[stop_sequence] = event_time


def handle_bus_arrival(
    event_time,
    stop_sequence,
    trip_id,
    stop_queues,
    bus_occupancy,
    waiting_records,
    queue_history,
    bus_history,
    occupancy_lookup,
    service_alighting_lookup,
    capacity=bus_capacity
):
    # Set the starting occupancy when the bus first enters the selected segment.
    if trip_id not in bus_occupancy:
        bus_occupancy[trip_id] = int(
            occupancy_lookup[trip_id]
        )

    occupancy_before = bus_occupancy[trip_id]

    # Remove assigned alightings, but never remove more passengers than are onboard.
    assigned_alightings = int(
        service_alighting_lookup[(trip_id, stop_sequence)]
    )

    actual_alightings = min(
        assigned_alightings,
        bus_occupancy[trip_id]
    )

    bus_occupancy[trip_id] -= actual_alightings

    # Board waiting passengers in FIFO order while capacity is available.
    available_capacity = capacity - bus_occupancy[trip_id]

    boarded = min(
        len(stop_queues[stop_sequence]),
        available_capacity
    )

    for _ in range(boarded):
        passenger_id, passenger_arrival = stop_queues[
            stop_sequence
        ].pop(0)

        waiting_time = event_time - passenger_arrival

        waiting_records.append({
            "Passenger ID": passenger_id,
            "Stop Sequence": stop_sequence,
            "Trip ID": trip_id,
            "Passenger Arrival": passenger_arrival,
            "Boarding Time": event_time,
            "Waiting Time": waiting_time
        })

    bus_occupancy[trip_id] += boarded
    remaining_queue = len(stop_queues[stop_sequence])

    queue_history.append({
        "Time": event_time,
        "Stop Sequence": stop_sequence,
        "Queue Length": remaining_queue,
        "Event": "bus_arrival"
    })

    bus_history.append({
        "Trip ID": trip_id,
        "Stop Sequence": stop_sequence,
        "Time": event_time,
        "Occupancy Before": occupancy_before,
        "Assigned Alightings": assigned_alightings,
        "Actual Alightings": actual_alightings,
        "Boarded": boarded,
        "Occupancy After": bus_occupancy[trip_id],
        "Passengers Left Waiting": remaining_queue
    })

In [17]:
def create_baseline_period_event_queues(passengers):
    morning_events = []
    evening_events = []

    # Add preserved passenger-arrival events.
    for _, passenger in passengers.iterrows():
        event = (
            float(passenger["Arrival Minutes"]),
            1,
            "passenger_arrival",
            int(passenger["Stop Sequence"]),
            passenger["Passenger ID"]
        )

        if passenger["Period"] == "Morning":
            heapq.heappush(morning_events, event)
        else:
            heapq.heappush(evening_events, event)

    # Add the fixed actual-GTFS bus events.
    for _, bus in study_stop_times.iterrows():
        event = (
            float(bus["arrival_minutes"]),
            0,
            "bus_arrival",
            int(bus["stop_sequence"]),
            bus["trip_id"]
        )

        if trip_period_lookup[bus["trip_id"]] == "Morning":
            heapq.heappush(morning_events, event)
        else:
            heapq.heappush(evening_events, event)

    return morning_events, evening_events

In [18]:
def run_service_simulation(
    period,
    events,
    generated_passengers,
    replication,
    seed,
    occupancy_lookup,
    service_alighting_lookup
):
    # Morning and evening are separate terminating simulations.
    if period == "Morning":
        start_time = 6 * 60
        end_time = 11 * 60
    else:
        start_time = 15 * 60
        end_time = 20 * 60

    simulation_events = events.copy()
    heapq.heapify(simulation_events)

    stop_queues = {stop: [] for stop in selected_stops}
    bus_occupancy = {}
    waiting_records = []
    queue_history = []
    bus_history = []

    peak_queue = {stop: 0 for stop in selected_stops}
    peak_queue_time = {stop: None for stop in selected_stops}

    simulation_clock = start_time
    period_end_queue = None
    period_end_recorded = False

    # Record the empty starting queue at every stop.
    for stop in selected_stops:
        queue_history.append({
            "Time": simulation_clock,
            "Stop Sequence": stop,
            "Queue Length": 0,
            "Event": "simulation_start"
        })

    while simulation_events:
        event = heapq.heappop(simulation_events)

        event_time, priority, event_type, stop_sequence, event_id = event

        # Record the official period-end queue after all events at the exact
        # boundary time are processed.
        if (not period_end_recorded) and (event_time > end_time):
            period_end_queue = {
                stop: len(queue)
                for stop, queue in stop_queues.items()
            }
            period_end_recorded = True

        simulation_clock = event_time

        if event_type == "passenger_arrival":
            handle_passenger_arrival(
                event_time,
                stop_sequence,
                event_id,
                stop_queues,
                queue_history,
                peak_queue,
                peak_queue_time
            )
        else:
            handle_bus_arrival(
                event_time,
                stop_sequence,
                event_id,
                stop_queues,
                bus_occupancy,
                waiting_records,
                queue_history,
                bus_history,
                occupancy_lookup,
                service_alighting_lookup,
                capacity=bus_capacity
            )

    if not period_end_recorded:
        period_end_queue = {
            stop: len(queue)
            for stop, queue in stop_queues.items()
        }

    final_backlog = {
        stop: len(queue)
        for stop, queue in stop_queues.items()
    }

    waiting_df = pd.DataFrame(waiting_records)
    bus_history_df = pd.DataFrame(bus_history)
    queue_history_df = pd.DataFrame(queue_history)

    completed_passengers = len(waiting_df)
    total_final_backlog = sum(final_backlog.values())

    passenger_conservation = (
        generated_passengers
        == completed_passengers + total_final_backlog
    )

    mean_wait = waiting_df["Waiting Time"].mean()
    median_wait = waiting_df["Waiting Time"].median()
    p95_wait = waiting_df["Waiting Time"].quantile(0.95)
    max_wait = waiting_df["Waiting Time"].max()

    overall_peak_queue = max(peak_queue.values())
    overall_peak_stop = max(peak_queue, key=peak_queue.get)
    overall_peak_time = peak_queue_time[overall_peak_stop]

    result = {
        "Replication": int(replication),
        "Seed": int(seed),
        "Period": period,
        "Generated Passengers": int(generated_passengers),
        "Completed Passengers": int(completed_passengers),
        "Mean Completed Wait": float(mean_wait),
        "Median Completed Wait": float(median_wait),
        "P95 Completed Wait": float(p95_wait),
        "Maximum Completed Wait": float(max_wait),
        "Peak Queue": int(overall_peak_queue),
        "Peak Queue Stop": int(overall_peak_stop),
        "Peak Queue Time": float(overall_peak_time),
        "Period-End Queue": int(sum(period_end_queue.values())),
        "Final Backlog": int(total_final_backlog),
        "Mean Bus Occupancy": float(bus_history_df["Occupancy After"].mean()),
        "Maximum Bus Occupancy": int(bus_history_df["Occupancy After"].max()),
        "Passenger Conservation": bool(passenger_conservation)
    }

    return result, waiting_df, bus_history_df, queue_history_df

## 5. Run the Baseline Replications

In [19]:
# Project Deliverable 1 starts with 30 independent seed sets.
# NumPy SeedSequence.spawn() creates reproducible child random streams.
# https://numpy.org/doc/stable/reference/random/parallel.html

seed_sequence = np.random.SeedSequence(master_seed)
child_sequences = seed_sequence.spawn(base_replications)

replication_seeds = [
    int(child.generate_state(1)[0])
    for child in child_sequences
]

seed_table = pd.DataFrame({
    "Replication": range(1, base_replications + 1),
    "Seed": replication_seeds
})

seed_table.head(10)

,Replication,Seed
0,1,479243620
1,2,454024514
2,3,1818751028
3,4,2288458700
4,5,2115331525
5,6,2666256175
6,7,975381835
7,8,2535166222
8,9,1079577930
9,10,3715024065


In [20]:
# Generate and preserve all 30 baseline passenger streams.
passenger_streams = []

for replication, seed in zip(
    seed_table["Replication"],
    seed_table["Seed"]
):
    passenger_streams.append(
        generate_baseline_passengers(
            seed=int(seed),
            replication=int(replication)
        )
    )

master_passenger_streams = pd.concat(
    passenger_streams,
    ignore_index=True
)

print("Replications saved:", master_passenger_streams["Replication"].nunique())
print("Passenger records saved:", len(master_passenger_streams))
print(
    "Passengers per replication:",
    master_passenger_streams.groupby("Replication").size().unique()
)

# Store each replication once so candidate evaluations do not repeatedly
# filter the complete passenger table.
passengers_by_replication = {
    int(replication): group.copy()
    for replication, group in master_passenger_streams.groupby("Replication")
}

Replications saved: 30
Passenger records saved: 23280
Passengers per replication: [776]


In [21]:
# Run the actual-GTFS baseline for all 30 passenger streams.
baseline_results = []

for replication, seed in zip(
    seed_table["Replication"],
    seed_table["Seed"]
):
    passengers = master_passenger_streams[
        master_passenger_streams["Replication"] == replication
    ].copy()

    morning_events, evening_events = create_baseline_period_event_queues(
        passengers
    )

    morning_passengers = int(
        (passengers["Period"] == "Morning").sum()
    )
    evening_passengers = int(
        (passengers["Period"] == "Evening").sum()
    )

    morning_result, _, _, _ = run_service_simulation(
        period="Morning",
        events=morning_events,
        generated_passengers=morning_passengers,
        replication=replication,
        seed=seed,
        occupancy_lookup=initial_occupancy_lookup,
        service_alighting_lookup=alighting_lookup
    )

    evening_result, _, _, _ = run_service_simulation(
        period="Evening",
        events=evening_events,
        generated_passengers=evening_passengers,
        replication=replication,
        seed=seed,
        occupancy_lookup=initial_occupancy_lookup,
        service_alighting_lookup=alighting_lookup
    )

    baseline_results.extend(
        [morning_result, evening_result]
    )

baseline_results_df = pd.DataFrame(baseline_results)

print("Baseline result rows:", len(baseline_results_df))
print(
    "All passenger conservation checks passed:",
    baseline_results_df["Passenger Conservation"].all()
)

baseline_results_df.head()

Baseline result rows: 60
All passenger conservation checks passed: True


,Replication,Seed,Period,Generated Passengers,Completed Passengers,Mean Completed Wait,Median Completed Wait,P95 Completed Wait,Maximum Completed Wait,Peak Queue,Peak Queue Stop,Peak Queue Time,Period-End Queue,Final Backlog,Mean Bus Occupancy,Maximum Bus Occupancy,Passenger Conservation
0,1,479243620,Morning,500,497,3.528452,3.034578,8.760788,12.785716,16,10,403.871403,5,3,15.902326,28,True
1,1,479243620,Evening,276,273,3.608862,3.200341,7.508045,21.203373,10,10,921.909650,3,3,12.231818,24,True
2,2,454024514,Morning,500,500,3.555722,3.047459,9.438976,13.712288,18,10,468.978623,3,0,15.972093,28,True
3,2,454024514,Evening,276,274,3.629559,3.486543,7.433749,16.877333,10,10,919.849591,2,2,12.245455,24,True
4,3,1818751028,Morning,500,499,3.747295,3.242815,8.535083,17.113319,18,10,468.613961,2,1,15.948837,29,True


In [22]:
# Law (2024) supports confidence-interval analysis across independent simulation
# replications. Project Deliverable 1 uses a 95% confidence interval and starts
# with 30 replications.
#
# Reference: SciPy Student t distribution
# https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.t.html

precision_records = []

for period in ["Morning", "Evening"]:
    period_results = baseline_results_df[
        baseline_results_df["Period"] == period
    ]

    for metric in ["Mean Completed Wait", "Peak Queue"]:
        values = period_results[metric]

        n = len(values)
        mean_value = values.mean()
        std_value = values.std(ddof=1)
        standard_error = std_value / np.sqrt(n)

        t_critical = t.ppf(0.975, df=n - 1)
        half_width = t_critical * standard_error

        lower_ci = mean_value - half_width
        upper_ci = mean_value + half_width

        if mean_value < 10:
            required_half_width = 1.0
        else:
            required_half_width = 0.10 * mean_value

        precision_records.append({
            "Period": period,
            "Metric": metric,
            "Replications": n,
            "Mean": mean_value,
            "Standard Deviation": std_value,
            "CI Lower": lower_ci,
            "CI Upper": upper_ci,
            "Half-Width": half_width,
            "Required Half-Width": required_half_width,
            "Precision Passed": half_width <= required_half_width
        })

baseline_precision = pd.DataFrame(precision_records)

baseline_precision.round(3)

,Period,Metric,Replications,Mean,Standard Deviation,CI Lower,CI Upper,Half-Width,Required Half-Width,Precision Passed
0,Morning,Mean Completed Wait,30,3.657,0.110,3.616,3.698,0.041,1.000,True
1,Morning,Peak Queue,30,17.267,1.999,16.520,18.013,0.746,1.727,True
2,Evening,Mean Completed Wait,30,3.651,0.164,3.590,3.712,0.061,1.000,True
3,Evening,Peak Queue,30,9.867,1.570,9.280,10.453,0.586,1.000,True


In [23]:
# Save the completed baseline inputs and outputs.
master_passenger_streams.to_csv(
    processed_dir / "M66_Master_Passenger_Streams_30_Replications.csv",
    index=False
)

seed_table.to_csv(
    processed_dir / "M66_Replication_Seeds.csv",
    index=False
)

baseline_results_df.to_csv(
    processed_dir / "M66_Baseline_30_Replications.csv",
    index=False
)

baseline_precision.to_csv(
    processed_dir / "M66_Baseline_Precision_Check.csv",
    index=False
)

print(
    "30 replications sufficient:",
    baseline_precision["Precision Passed"].all()
)

30 replications sufficient: True


## 6. Build Candidate Service Plans for AI Evaluation

In [24]:
# Candidate service plans keep the same total number of buses as the baseline
# during the AI technique comparison. The optimizer changes the hourly allocation.
#
# Oliveira et al. (2024) include minimum and maximum departure frequencies in a
# bus frequency-setting model.
# https://doi.org/10.1016/j.asoc.2024.111578

min_hourly_trips = int(baseline_service_plan["Trips"].min())
max_hourly_trips = int(baseline_service_plan["Trips"].max())

baseline_plan_lookup = {
    period: tuple(
        baseline_service_plan[
            baseline_service_plan["Period"] == period
        ]["Trips"].astype(int).tolist()
    )
    for period in ["Morning", "Evening"]
}

print("Minimum trips per hour:", min_hourly_trips)
print("Maximum trips per hour:", max_hourly_trips)
print("Morning baseline plan:", baseline_plan_lookup["Morning"])
print("Evening baseline plan:", baseline_plan_lookup["Evening"])

Minimum trips per hour: 5
Maximum trips per hour: 12
Morning baseline plan: (5, 12, 12, 8, 6)
Evening baseline plan: (8, 12, 11, 8, 5)


In [25]:
# Use the GTFS schedule to measure the typical travel time from Stop 6 to each
# downstream stop during each study hour.
trip_times = study_stop_times[
    ["trip_id", "stop_sequence", "arrival_minutes"]
].merge(
    trip_initial_occupancy[["trip_id", "Hour"]],
    on="trip_id",
    how="left"
)

stop6_times = (
    trip_times[
        trip_times["stop_sequence"] == 6
    ][["trip_id", "arrival_minutes"]]
    .rename(columns={"arrival_minutes": "stop6_arrival"})
)

trip_times = trip_times.merge(
    stop6_times,
    on="trip_id",
    how="left"
)

trip_times["Travel Offset"] = (
    trip_times["arrival_minutes"]
    - trip_times["stop6_arrival"]
)

hourly_travel_offsets = (
    trip_times
    .groupby(["Hour", "stop_sequence"])["Travel Offset"]
    .median()
    .reset_index()
    .rename(
        columns={
            "stop_sequence": "Stop Sequence",
            "Travel Offset": "Median Offset Minutes"
        }
    )
)

travel_offset_lookup = (
    hourly_travel_offsets
    .set_index(["Hour", "Stop Sequence"])["Median Offset Minutes"]
    .to_dict()
)

print("Hour-stop travel offsets:", len(travel_offset_lookup))
hourly_travel_offsets.head(10)

Hour-stop travel offsets: 50


,Hour,Stop Sequence,Median Offset Minutes
0,6,6,0.000000
1,6,7,3.516667
2,6,8,4.100000
3,6,9,5.466667
4,6,10,9.100000
5,7,6,0.000000
6,7,7,4.933333
7,7,8,5.741667
8,7,9,7.375000
9,7,10,11.741667


## 7. Generalized Optimization Layer (Scenario Context)

The scenario context is a single dictionary that carries everything the optimizer needs for one operating condition. Baseline builds a context from the reused baseline objects. High-Demand and Reduced-Service will later build their own contexts from Karol's scenario inputs, and the same optimizer code runs unchanged.

In [26]:
# (generalized/extended from baseline) Package every scenario-varying input the
# GA needs into one context dict. Baseline, High-Demand, and Reduced-Service
# differ only in this context, not in the optimizer code below.
def make_scenario_context(
    name,
    passengers_by_replication,
    seed_table,
    service_budget,
    min_hourly_trips,
    max_hourly_trips,
    reference_plan_lookup,
    search_replication_ids,
    validation_replication_ids,
    upstream_flow_lookup,
    alighting_targets,
    travel_offset_lookup,
    feasibility_backlog
):
    return {
        "name": name,
        "passengers_by_replication": passengers_by_replication,
        "seed_lookup": seed_table.set_index("Replication")["Seed"].to_dict(),
        "service_budget": dict(service_budget),
        "min_hourly_trips": int(min_hourly_trips),
        "max_hourly_trips": int(max_hourly_trips),
        "reference_plan_lookup": dict(reference_plan_lookup),
        "search_replication_ids": list(search_replication_ids),
        "validation_replication_ids": list(validation_replication_ids),
        "upstream_flow_lookup": dict(upstream_flow_lookup),
        "alighting_targets": alighting_targets.copy(),
        "travel_offset_lookup": dict(travel_offset_lookup),
        "feasibility_backlog": dict(feasibility_backlog),
        "fitness_cache": {}
    }

In [27]:
# (generalized/extended from baseline) Candidate schedule builders. Same logic
# as the baseline notebook, but the travel offsets, upstream flow, and alighting
# targets are read from the scenario context so each scenario uses its own inputs.
# Reference: Gkiotsalitis (2020), headway-based bus scheduling and waiting time.
# https://doi.org/10.1049/iet-its.2019.0725
def build_candidate_stop6_schedule(ctx, period, trip_counts):
    hours = morning_hours if period == "Morning" else evening_hours
    records = []
    for hour, count in zip(hours, trip_counts):
        count = int(count)
        if count < 1:
            raise ValueError("Each study hour must contain at least one bus.")
        headway = 60 / count
        within_hour_times = (np.arange(count) + 0.5) * headway
        for trip_number, minute_offset in enumerate(within_hour_times, start=1):
            records.append({
                "trip_id": f"CAND_{period}_{hour:02d}_{trip_number:02d}",
                "Hour": hour,
                "Period": period,
                "Stop 6 Arrival": hour * 60 + minute_offset,
                "Hourly Trips": count,
                "Headway Minutes": headway
            })
    return pd.DataFrame(records)


def expand_candidate_stop_times(ctx, candidate_schedule):
    travel_offset_lookup = ctx["travel_offset_lookup"]
    records = []
    for _, trip in candidate_schedule.iterrows():
        hour = int(trip["Hour"])
        for stop in selected_stops:
            records.append({
                "trip_id": trip["trip_id"],
                "Hour": hour,
                "Period": trip["Period"],
                "stop_sequence": stop,
                "arrival_minutes": (
                    float(trip["Stop 6 Arrival"])
                    + float(travel_offset_lookup[(hour, stop)])
                )
            })
    return pd.DataFrame(records)


def create_candidate_initial_occupancy(ctx, candidate_schedule):
    upstream_flow_lookup = ctx["upstream_flow_lookup"]
    candidate_hourly_counts = candidate_schedule.groupby("Hour").size().to_dict()
    records = []
    for _, trip in candidate_schedule.iterrows():
        hour = int(trip["Hour"])
        bus_count = candidate_hourly_counts[hour]
        unrounded = upstream_flow_lookup[hour] / bus_count
        initial_occupancy = int(
            np.minimum(60, np.maximum(0, np.floor(unrounded + 0.5)))
        )
        records.append({
            "trip_id": trip["trip_id"],
            "Hour": hour,
            "Period": trip["Period"],
            "Initial Occupancy": initial_occupancy
        })
    return pd.DataFrame(records)


def create_candidate_alighting_plan(ctx, candidate_stop_times):
    alighting_targets = ctx["alighting_targets"]
    plan = candidate_stop_times.merge(
        alighting_targets[["Stop Sequence", "Hour", "Alighting Target"]],
        left_on=["stop_sequence", "Hour"],
        right_on=["Stop Sequence", "Hour"],
        how="left"
    )
    plan = plan.sort_values(["stop_sequence", "Hour", "arrival_minutes"]).reset_index(drop=True)
    plan["Assigned Alightings"] = 0
    for (stop, hour), group in plan.groupby(["stop_sequence", "Hour"]):
        target = int(group["Alighting Target"].iloc[0])
        bus_count = len(group)
        quotient = target // bus_count
        remainder = target % bus_count
        assignments = [quotient + 1] * remainder
        assignments += [quotient] * (bus_count - remainder)
        plan.loc[group.index, "Assigned Alightings"] = assignments
    return plan


def create_candidate_event_queue(passengers, candidate_stop_times, period):
    # Unchanged from baseline. Builds the chronological event queue for a period.
    events = []
    period_passengers = passengers[passengers["Period"] == period]
    for _, passenger in period_passengers.iterrows():
        heapq.heappush(events, (
            float(passenger["Arrival Minutes"]), 1, "passenger_arrival",
            int(passenger["Stop Sequence"]), passenger["Passenger ID"]
        ))
    for _, bus in candidate_stop_times.iterrows():
        heapq.heappush(events, (
            float(bus["arrival_minutes"]), 0, "bus_arrival",
            int(bus["stop_sequence"]), bus["trip_id"]
        ))
    return events

In [28]:
# (generalized/extended from baseline) Plan validity and candidate evaluation.
# The fixed constraints (5 study hours, per-hour trip bounds, fixed budget) and
# the passenger streams are now read from the scenario context.
def is_valid_service_plan(ctx, period, trip_counts):
    trip_counts = [int(x) for x in trip_counts]
    if len(trip_counts) != 5:
        return False
    if any(c < ctx["min_hourly_trips"] or c > ctx["max_hourly_trips"] for c in trip_counts):
        return False
    if sum(trip_counts) != ctx["service_budget"][period]:
        return False
    return True


def evaluate_candidate_replication(ctx, period, trip_counts, passengers, replication, seed):
    trip_counts = [int(x) for x in trip_counts]
    if not is_valid_service_plan(ctx, period, trip_counts):
        raise ValueError("The candidate service plan does not meet the fixed constraints.")
    candidate_schedule = build_candidate_stop6_schedule(ctx, period, trip_counts)
    candidate_stop_times = expand_candidate_stop_times(ctx, candidate_schedule)
    candidate_occupancy = create_candidate_initial_occupancy(ctx, candidate_schedule)
    candidate_alighting = create_candidate_alighting_plan(ctx, candidate_stop_times)
    occupancy_lookup = candidate_occupancy.set_index("trip_id")["Initial Occupancy"].to_dict()
    candidate_alighting_lookup = candidate_alighting.set_index(["trip_id", "stop_sequence"])["Assigned Alightings"].to_dict()
    candidate_events = create_candidate_event_queue(passengers, candidate_stop_times, period)
    generated_passengers = int((passengers["Period"] == period).sum())
    result, _, _, _ = run_service_simulation(
        period=period,
        events=candidate_events,
        generated_passengers=generated_passengers,
        replication=replication,
        seed=seed,
        occupancy_lookup=occupancy_lookup,
        service_alighting_lookup=candidate_alighting_lookup
    )
    result["Service Plan"] = tuple(trip_counts)
    return result


def evaluate_candidate_replications(ctx, period, trip_counts, replication_ids):
    results = []
    for replication in replication_ids:
        seed = int(ctx["seed_lookup"][int(replication)])
        passengers = ctx["passengers_by_replication"][int(replication)]
        results.append(evaluate_candidate_replication(
            ctx, period, trip_counts, passengers, int(replication), seed
        ))
    return pd.DataFrame(results)

In [29]:
# (generalized/extended from baseline) Fitness objective. Lower mean completed
# waiting time is better. A plan is feasible only when its mean final backlog
# does not exceed the scenario's own reference backlog (the final-backlog
# safeguard). The cache lives inside the context so scenarios never mix results.
# Reference: Oliveira et al. (2024), waiting-time objective in bus frequency setting.
# https://doi.org/10.1016/j.asoc.2024.111578
def evaluate_service_plan_fitness(ctx, period, trip_counts):
    plan = tuple(int(x) for x in trip_counts)
    key = (period, plan)
    cache = ctx["fitness_cache"]
    if key not in cache:
        results = evaluate_candidate_replications(
            ctx, period, list(plan), ctx["search_replication_ids"]
        )
        mean_wait = results["Mean Completed Wait"].mean()
        mean_backlog = results["Final Backlog"].mean()
        mean_peak_queue = results["Peak Queue"].mean()
        mean_p95_wait = results["P95 Completed Wait"].mean()
        feasible = mean_backlog <= ctx["feasibility_backlog"][period]
        cache[key] = {
            "Period": period,
            "Service Plan": plan,
            "Fitness": float(mean_wait) if feasible else np.inf,
            "Mean Wait": float(mean_wait),
            "Mean Backlog": float(mean_backlog),
            "Mean Peak Queue": float(mean_peak_queue),
            "Mean P95 Wait": float(mean_p95_wait),
            "Feasible": bool(feasible)
        }
    return cache[key]

In [30]:
# (generalized/extended from baseline) Helper functions used by the GA. Same
# logic as the baseline notebook, with the bounds and budget taken from context.
def random_valid_plan(ctx, period, rng):
    budget = int(ctx["service_budget"][period])
    lo, hi = ctx["min_hourly_trips"], ctx["max_hourly_trips"]
    while True:
        plan = rng.integers(lo, hi + 1, size=5)
        if int(plan.sum()) == budget:
            return tuple(int(x) for x in plan)


def repair_service_plan(ctx, period, values, rng):
    lo, hi = ctx["min_hourly_trips"], ctx["max_hourly_trips"]
    budget = int(ctx["service_budget"][period])
    plan = np.rint(np.array(values, dtype=float)).astype(int)
    plan = np.clip(plan, lo, hi)
    while plan.sum() < budget:
        choices = np.where(plan < hi)[0]
        plan[int(rng.choice(choices))] += 1
    while plan.sum() > budget:
        choices = np.where(plan > lo)[0]
        plan[int(rng.choice(choices))] -= 1
    return tuple(int(x) for x in plan)


def neighboring_service_plan(ctx, period, plan, rng):
    lo, hi = ctx["min_hourly_trips"], ctx["max_hourly_trips"]
    plan = np.array(plan, dtype=int)
    donors = np.where(plan > lo)[0]
    receivers = np.where(plan < hi)[0]
    for _ in range(30):
        donor = int(rng.choice(donors))
        receiver = int(rng.choice(receivers))
        if donor != receiver:
            plan[donor] -= 1
            plan[receiver] += 1
            return tuple(int(x) for x in plan)
    return tuple(int(x) for x in plan)


def get_run_fitness(ctx, period, plan, seen_plans):
    plan = tuple(int(x) for x in plan)
    result = evaluate_service_plan_fitness(ctx, period, plan)
    seen_plans.add(plan)
    return result

In [31]:
# (generalized/extended from baseline) The Genetic Algorithm. The search logic,
# population size, tournament selection, two elites, uniform-mask crossover,
# budget-preserving repair, and optional neighbor mutation are all unchanged
# from the Midterm GA. Only the seed plan, bounds, and budget now come from ctx.
# Reference: Oliveira et al. (2024), genetic algorithm for bus frequency setting.
# https://doi.org/10.1016/j.asoc.2024.111578
def run_genetic_algorithm(ctx, period, optimizer_seed, evaluation_budget=30, population_size=6):
    rng = np.random.default_rng(optimizer_seed)
    population = [ctx["reference_plan_lookup"][period]]
    while len(population) < population_size:
        plan = random_valid_plan(ctx, period, rng)
        if plan not in population:
            population.append(plan)
    seen_plans = set()
    best_result = None
    history = []
    while len(seen_plans) < evaluation_budget:
        scored = []
        for plan in population:
            if plan not in seen_plans and len(seen_plans) >= evaluation_budget:
                break
            result = get_run_fitness(ctx, period, plan, seen_plans)
            scored.append((result["Fitness"], plan, result))
            if best_result is None or result["Fitness"] < best_result["Fitness"]:
                best_result = result
            if len(seen_plans) >= evaluation_budget:
                break
        history.append({
            "Technique": "GA",
            "Period": period,
            "Optimizer Seed": optimizer_seed,
            "Unique Plans Evaluated": len(seen_plans),
            "Best Fitness": best_result["Fitness"]
        })
        if len(seen_plans) >= evaluation_budget:
            break
        scored = sorted(scored, key=lambda item: item[0])
        elites = [scored[0][1], scored[min(1, len(scored) - 1)][1]]
        def tournament_select():
            indexes = rng.choice(len(scored), size=min(3, len(scored)), replace=False)
            return min([scored[int(i)] for i in indexes], key=lambda item: item[0])[1]
        next_population = elites.copy()
        while len(next_population) < population_size:
            parent1 = np.array(tournament_select())
            parent2 = np.array(tournament_select())
            mask = rng.random(5) < 0.5
            child = np.where(mask, parent1, parent2)
            child = repair_service_plan(ctx, period, child, rng)
            if rng.random() < 0.5:
                child = neighboring_service_plan(ctx, period, child, rng)
            next_population.append(child)
        population = next_population
    return best_result, pd.DataFrame(history)

In [32]:
# (new) Driver that runs the GA for one scenario across the optimizer seeds,
# validates each run's best plan on the full validation replications, and selects
# the final plan per period.
#
# QA item 1: when final_backlog_reference is provided, the final plan is selected
# only among validated plans whose 30-replication mean final backlog does not
# exceed the scenario reference over the same replications. Baseline passes None,
# so its selection stays identical to the Midterm reproduction.
# QA item 4: the selected summary keeps all required metrics, including
# Period-End Queue, Mean Bus Occupancy, and Maximum Bus Occupancy.
def run_ga_for_scenario(ctx, optimizer_seeds, evaluation_budget, population_size=6, final_backlog_reference=None):
    run_records = []
    for period in ["Morning", "Evening"]:
        for optimizer_seed in optimizer_seeds:
            search_best, _ = run_genetic_algorithm(
                ctx, period, optimizer_seed, evaluation_budget, population_size
            )
            validated = evaluate_candidate_replications(
                ctx, period, list(search_best["Service Plan"]), ctx["validation_replication_ids"]
            )
            run_records.append({
                "Scenario": ctx["name"], "Period": period, "Technique": "GA",
                "Optimizer Seed": optimizer_seed, "Evaluation Budget": evaluation_budget,
                "Search Best Fitness": search_best["Fitness"],
                "Best Service Plan": search_best["Service Plan"],
                "Validated Mean Wait": validated["Mean Completed Wait"].mean(),
                "Validated Mean Backlog": validated["Final Backlog"].mean(),
                "Validated Mean Peak Queue": validated["Peak Queue"].mean(),
                "Validated Mean P95 Wait": validated["P95 Completed Wait"].mean(),
                "Validated Mean Period-End Queue": validated["Period-End Queue"].mean(),
                "Validated Mean Bus Occupancy": validated["Mean Bus Occupancy"].mean(),
                "Validated Max Bus Occupancy": int(validated["Maximum Bus Occupancy"].max()),
                "Passenger Conservation": bool(validated["Passenger Conservation"].all())
            })
    runs_df = pd.DataFrame(run_records)

    selected_records = []
    for period in ["Morning", "Evening"]:
        period_runs = runs_df[runs_df["Period"] == period].copy()
        if final_backlog_reference is not None:
            feasible = period_runs[
                period_runs["Validated Mean Backlog"] <= final_backlog_reference[period] + 1e-9
            ]
            pool = feasible if len(feasible) > 0 else period_runs
            passed = len(feasible) > 0
        else:
            pool = period_runs
            passed = True
        best_row = pool.loc[pool["Validated Mean Wait"].idxmin()]
        selected_records.append({
            "Scenario": ctx["name"], "Period": period, "Technique": "GA",
            "Service Plan": best_row["Best Service Plan"],
            "Mean Wait": best_row["Validated Mean Wait"],
            "Mean Backlog": best_row["Validated Mean Backlog"],
            "Mean Peak Queue": best_row["Validated Mean Peak Queue"],
            "Mean P95 Wait": best_row["Validated Mean P95 Wait"],
            "Mean Period-End Queue": best_row["Validated Mean Period-End Queue"],
            "Mean Bus Occupancy": best_row["Validated Mean Bus Occupancy"],
            "Maximum Bus Occupancy": best_row["Validated Max Bus Occupancy"],
            "Passed Final Backlog Safeguard": bool(passed)
        })
    selected_df = pd.DataFrame(selected_records)
    return runs_df, selected_df

In [33]:
# (new) Detailed evaluation of one selected plan over the validation replications.
# Returns the full per-replication rows (QA item 2) with every required metric plus
# validation flags (QA item 5), and the stop-level rows (QA item 7). Uses the same
# candidate build and the same reused simulator as the GA, so the numbers match.
def evaluate_selected_plan_detailed(ctx, period, trip_counts, replication_ids):
    end_time = 11 * 60 if period == "Morning" else 20 * 60
    plan = [int(x) for x in trip_counts]
    per_rep_rows = []
    stop_rows = []
    for replication in replication_ids:
        seed = int(ctx["seed_lookup"][int(replication)])
        passengers = ctx["passengers_by_replication"][int(replication)]
        candidate_schedule = build_candidate_stop6_schedule(ctx, period, plan)
        candidate_stop_times = expand_candidate_stop_times(ctx, candidate_schedule)
        candidate_occupancy = create_candidate_initial_occupancy(ctx, candidate_schedule)
        candidate_alighting = create_candidate_alighting_plan(ctx, candidate_stop_times)
        occupancy_lookup = candidate_occupancy.set_index("trip_id")["Initial Occupancy"].to_dict()
        alighting_lookup = candidate_alighting.set_index(["trip_id", "stop_sequence"])["Assigned Alightings"].to_dict()
        events = create_candidate_event_queue(passengers, candidate_stop_times, period)
        generated = int((passengers["Period"] == period).sum())
        result, waiting_df, bus_history_df, queue_history_df = run_service_simulation(
            period=period, events=events, generated_passengers=generated,
            replication=int(replication), seed=seed,
            occupancy_lookup=occupancy_lookup, service_alighting_lookup=alighting_lookup
        )
        min_wait = float(waiting_df["Waiting Time"].min())
        min_occ = int(bus_history_df["Occupancy After"].min())
        max_occ = int(bus_history_df["Occupancy After"].max())
        row = dict(result)
        row["Scenario"] = ctx["name"]
        row["Service Plan"] = tuple(plan)
        row["Minimum Completed Wait"] = min_wait
        row["Minimum Bus Occupancy"] = min_occ
        row["Waiting Time Valid"] = bool(min_wait >= 0)
        row["Bus Occupancy Valid"] = bool(min_occ >= 0 and max_occ <= bus_capacity)
        per_rep_rows.append(row)
        for stop in selected_stops:
            waits = waiting_df[waiting_df["Stop Sequence"] == stop]["Waiting Time"]
            queue_stop = queue_history_df[queue_history_df["Stop Sequence"] == stop]
            queue_end = queue_stop[queue_stop["Time"] <= end_time]
            stop_rows.append({
                "Scenario": ctx["name"], "Period": period,
                "Replication": int(replication), "Seed": seed,
                "Stop Sequence": int(stop),
                "Mean Wait": float(waits.mean()) if len(waits) else 0.0,
                "P95 Wait": float(waits.quantile(0.95)) if len(waits) else 0.0,
                "Peak Queue": int(queue_stop["Queue Length"].max()) if len(queue_stop) else 0,
                "Period-End Queue": int(queue_end["Queue Length"].iloc[-1]) if len(queue_end) else 0,
                "Final Backlog": int(queue_stop["Queue Length"].iloc[-1]) if len(queue_stop) else 0
            })
    per_rep_df = pd.DataFrame(per_rep_rows)
    ordered = ["Scenario", "Replication", "Seed", "Period", "Service Plan",
               "Mean Completed Wait", "P95 Completed Wait", "Peak Queue", "Period-End Queue",
               "Final Backlog", "Mean Bus Occupancy", "Maximum Bus Occupancy", "Minimum Bus Occupancy",
               "Minimum Completed Wait", "Waiting Time Valid", "Bus Occupancy Valid", "Passenger Conservation"]
    per_rep_df = per_rep_df[[c for c in ordered if c in per_rep_df.columns]]
    stop_df = pd.DataFrame(stop_rows)
    return per_rep_df, stop_df


# (new) Run the detailed evaluation for both periods of one scenario's selected plans.
def evaluate_scenario_selected_plans(ctx, selected_df):
    per_rep_frames = []
    stop_frames = []
    for _, row in selected_df.iterrows():
        plan = tuple(int(x) for x in row["Service Plan"])
        per_rep, stop_level = evaluate_selected_plan_detailed(
            ctx, row["Period"], plan, ctx["validation_replication_ids"]
        )
        per_rep_frames.append(per_rep)
        stop_frames.append(stop_level)
    return pd.concat(per_rep_frames, ignore_index=True), pd.concat(stop_frames, ignore_index=True)

In [34]:
# (new) Replication precision check, same 95% confidence-interval rule used across
# the project (QA item 6). Half-width must be within 1.0 minute when the mean is
# below 10, otherwise within 10 percent of the mean.
# Reference: Law (2015), Simulation Modeling and Analysis, confidence-interval
# precision for terminating simulations.
precision_confidence_level = 0.95
precision_start_replications = base_replications
precision_increment = 10
precision_max_replications = 50

def compute_precision_check(results_df, scenario_name):
    records = []
    all_passed = True
    for period in ["Morning", "Evening"]:
        period_results = results_df[results_df["Period"] == period]
        for metric in ["Mean Completed Wait", "Peak Queue"]:
            values = period_results[metric]
            n = len(values)
            mean_value = values.mean()
            std_value = values.std(ddof=1)
            standard_error = std_value / np.sqrt(n)
            t_critical = t.ppf(precision_confidence_level + (1 - precision_confidence_level) / 2, df=n - 1)
            half_width = t_critical * standard_error
            required_half_width = 1.0 if mean_value < 10 else 0.10 * mean_value
            passed = half_width <= required_half_width
            all_passed = all_passed and passed
            records.append({
                "Scenario": scenario_name, "Period": period, "Metric": metric,
                "Replications": n, "Mean": round(float(mean_value), 4),
                "Half-Width": round(float(half_width), 4),
                "Required Half-Width": round(float(required_half_width), 4),
                "Precision Passed": bool(passed)
            })
    return pd.DataFrame(records), all_passed

In [35]:
# (new) Paired reference-versus-GA comparison with a t-based 95% confidence
# interval (QA item 3). Reference and GA rows are matched on Replication, Seed,
# and Period, then GA minus Reference is computed per replication. An improvement
# is only called statistically supported when the whole 95% CI is below zero.
def paired_reference_vs_ga(ga_per_rep, reference_df, scenario_name, metrics, confidence=0.95):
    rows = []
    for period in ["Morning", "Evening"]:
        g = ga_per_rep[ga_per_rep["Period"] == period]
        r = reference_df[reference_df["Period"] == period]
        for metric in metrics:
            merged = g[["Replication", "Seed", metric]].merge(
                r[["Replication", "Seed", metric]],
                on=["Replication", "Seed"], suffixes=(" GA", " Ref")
            )
            diff = merged[metric + " GA"] - merged[metric + " Ref"]
            n = len(diff)
            mean_d = diff.mean()
            sd = diff.std(ddof=1)
            se = sd / np.sqrt(n)
            t_critical = t.ppf(confidence + (1 - confidence) / 2, df=n - 1)
            half_width = t_critical * se
            rows.append({
                "Scenario": scenario_name, "Period": period, "Metric": metric, "N": n,
                "Mean GA": round(float(merged[metric + " GA"].mean()), 4),
                "Mean Reference": round(float(merged[metric + " Ref"].mean()), 4),
                "Mean Paired Diff (GA - Reference)": round(float(mean_d), 4),
                "CI Lower": round(float(mean_d - half_width), 4),
                "CI Upper": round(float(mean_d + half_width), 4),
                "Statistically Supported Improvement": bool((mean_d + half_width) < 0)
            })
    return pd.DataFrame(rows)

## 8. Baseline Verification

Before any scenario is integrated, the generalized GA is run on a Baseline context built from the reused baseline objects. The selected Morning and Evening plans and their validated mean waiting times are compared against the Midterm result in `selected_ai_service_plans.csv`. Matching values confirm the refactor did not change the GA behavior.

In [36]:
# Build the Baseline feasibility reference: the mean final backlog of the actual
# GTFS baseline over the search replications, used as the safeguard threshold.
baseline_feasibility_backlog = (
    baseline_results_df[
        baseline_results_df["Replication"].isin(optimization_replications)
    ]
    .groupby("Period")["Final Backlog"]
    .mean()
    .to_dict()
)

# Upstream net-flow lookup used for candidate initial occupancy.
baseline_upstream_flow_lookup = (
    upstream_hourly_flow.set_index("Hour")["Upstream Net Flow"].to_dict()
)

baseline_ctx = make_scenario_context(
    name="Baseline",
    passengers_by_replication=passengers_by_replication,
    seed_table=seed_table,
    service_budget=service_budget,
    min_hourly_trips=min_hourly_trips,
    max_hourly_trips=max_hourly_trips,
    reference_plan_lookup=baseline_plan_lookup,
    search_replication_ids=optimization_replications,
    validation_replication_ids=range(1, base_replications + 1),
    upstream_flow_lookup=baseline_upstream_flow_lookup,
    alighting_targets=hourly_alighting_targets,
    travel_offset_lookup=travel_offset_lookup,
    feasibility_backlog=baseline_feasibility_backlog
)

print("Baseline context ready.")
print("Service budget:", baseline_ctx["service_budget"])
print("Hourly trip bounds:", baseline_ctx["min_hourly_trips"], "to", baseline_ctx["max_hourly_trips"])
print("Feasibility backlog threshold:", {k: round(v, 3) for k, v in baseline_ctx["feasibility_backlog"].items()})

Baseline context ready.
Service budget: {'Evening': 44, 'Morning': 43}
Hourly trip bounds: 5 to 12
Feasibility backlog threshold: {'Evening': 2.4, 'Morning': 1.2}


In [37]:
# Run the generalized GA on Baseline using the same optimizer seeds and
# evaluation budget as the Midterm screening.
ga_core_runs, ga_core_selected = run_ga_for_scenario(
    baseline_ctx,
    optimizer_seeds=optimizer_seeds,
    evaluation_budget=evaluation_budget
)

ga_core_selected

,Scenario,Period,Technique,Service Plan,Mean Wait,Mean Backlog,Mean Peak Queue,Mean P95 Wait,Mean Period-End Queue,Mean Bus Occupancy,Maximum Bus Occupancy,Passed Final Backlog Safeguard
0,Baseline,Morning,GA,"(8, 9, 11, 9, 6)",3.488068,1.833333,17.500000,7.146191,2.933333,15.673643,36,True
1,Baseline,Evening,GA,"(11, 7, 9, 10, 7)",3.532199,0.966667,8.633333,7.473238,2.166667,12.119091,28,True


In [38]:
# Compare the generalized GA result against the Midterm GA result.
# QA item 11: use ast.literal_eval instead of eval for the stored plan tuple.
# Reference: pandas.read_csv()
# https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html
import ast

midterm_selected = pd.read_csv(tables_dir / "selected_ai_service_plans.csv")

check_rows = []
for period in ["Morning", "Evening"]:
    new_row = ga_core_selected[ga_core_selected["Period"] == period].iloc[0]
    old_row = midterm_selected[midterm_selected["Period"] == period].iloc[0]
    new_plan = tuple(int(x) for x in new_row["Service Plan"])
    old_plan = tuple(int(x) for x in ast.literal_eval(old_row["Service Plan"]))
    check_rows.append({
        "Period": period,
        "Generalized Plan": new_plan,
        "Midterm Plan": old_plan,
        "Plan Matches": new_plan == old_plan,
        "Generalized Mean Wait": round(float(new_row["Mean Wait"]), 6),
        "Midterm Mean Wait": round(float(old_row["Mean Wait"]), 6),
        "Wait Matches": bool(np.isclose(float(new_row["Mean Wait"]), float(old_row["Mean Wait"])))
    })

baseline_reproduction_check = pd.DataFrame(check_rows)
baseline_reproduction_ok = bool(
    baseline_reproduction_check["Plan Matches"].all()
    and baseline_reproduction_check["Wait Matches"].all()
)
print("Generalized GA reproduces the Midterm GA result on Baseline:", baseline_reproduction_ok)
baseline_reproduction_check

Generalized GA reproduces the Midterm GA result on Baseline: True


,Period,Generalized Plan,Midterm Plan,Plan Matches,Generalized Mean Wait,Midterm Mean Wait,Wait Matches
0,Morning,"(8, 9, 11, 9, 6)","(8, 9, 11, 9, 6)",True,3.488068,3.488068,True
1,Evening,"(11, 7, 9, 10, 7)","(11, 7, 9, 10, 7)",True,3.532199,3.532199,True


In [39]:
# QA item 8: fail-fast. Stop the notebook if the generalized GA does not
# reproduce the Midterm Baseline result, before any final outputs are saved.
if not baseline_reproduction_ok:
    raise RuntimeError(
        "Baseline reproduction failed: the generalized GA did not reproduce the "
        "Midterm Baseline plans or mean waits. Stopping before saving outputs."
    )
print("Baseline reproduction gate passed.")

Baseline reproduction gate passed.


In [40]:
# Save the GA-core verification outputs under new filenames so the Midterm
# tables are not overwritten.
ga_core_runs_export = ga_core_runs.copy()
ga_core_runs_export["Best Service Plan"] = ga_core_runs_export["Best Service Plan"].astype(str)
ga_core_runs_export.to_csv(tables_dir / "ga_core_baseline_runs.csv", index=False)

ga_core_selected_export = ga_core_selected.copy()
ga_core_selected_export["Service Plan"] = ga_core_selected_export["Service Plan"].astype(str)
ga_core_selected_export.to_csv(tables_dir / "ga_core_baseline_selected_plans.csv", index=False)

baseline_reproduction_check.to_csv(tables_dir / "ga_core_baseline_reproduction_check.csv", index=False)

print("Saved GA-core Baseline outputs to", tables_dir)

Saved GA-core Baseline outputs to ../output/tables


## 9. Scenario Integration — High-Demand and Reduced-Service

This section feeds Karol's Pre-Final scenario inputs into the same generalized GA. Each scenario builds its own context and reuses the optimizer code unchanged. High-Demand keeps the Baseline service budget, bounds, and reference plan because only passenger demand changes. Reduced-Service uses the smaller service budget, bounds, and reference plan from Karol's reduced schedule. In both cases the GA only redistributes the fixed service budget for that scenario. Upstream initial occupancy is left unchanged across scenarios, matching the documented scope of the scenario notebook.

Final plan selection uses the final-backlog safeguard on the full validation replications: after validating every optimizer seed's best plan on all 30 replications, any plan whose mean final backlog exceeds the scenario reference over the same replications is excluded, and the lowest validated mean wait is chosen only among the plans that remain.

In [41]:
# (new) Load Karol's Pre-Final scenario handoff files from the shared repo.
# Reference: pandas.read_csv()
# https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html
high_demand_streams = pd.read_csv(processed_dir / "M66_HighDemand_Master_Passenger_Streams.csv")
high_demand_results = pd.read_csv(processed_dir / "M66_HighDemand_Results.csv")
reduced_service_results = pd.read_csv(processed_dir / "M66_ReducedService_Results.csv")
reduced_service_plan_in = pd.read_csv(processed_dir / "M66_ReducedService_Service_Plan.csv")

# High-Demand passenger streams grouped into the same per-replication form the
# optimizer expects: one DataFrame per replication, both periods included.
high_demand_passengers_by_replication = {
    int(rep): group.reset_index(drop=True)
    for rep, group in high_demand_streams.groupby("Replication")
}
print("High-Demand replications loaded:", len(high_demand_passengers_by_replication))

High-Demand replications loaded: 30


In [42]:
# (new) High-Demand context. Service is unchanged, so the budget, bounds, and
# reference plan are the Baseline ones. Demand comes from Karol's High-Demand
# streams. The search-time feasibility threshold uses the first 5 matched
# replications; the selection-time safeguard uses all 30.
high_demand_feasibility_backlog = (
    high_demand_results[high_demand_results["Replication"].isin(optimization_replications)]
    .groupby("Period")["Final Backlog"].mean().to_dict()
)
high_demand_reference_backlog_validation = (
    high_demand_results[high_demand_results["Replication"].isin(range(1, base_replications + 1))]
    .groupby("Period")["Final Backlog"].mean().to_dict()
)

high_demand_ctx = make_scenario_context(
    name="High-Demand",
    passengers_by_replication=high_demand_passengers_by_replication,
    seed_table=seed_table,
    service_budget=service_budget,
    min_hourly_trips=min_hourly_trips,
    max_hourly_trips=max_hourly_trips,
    reference_plan_lookup=baseline_plan_lookup,
    search_replication_ids=optimization_replications,
    validation_replication_ids=range(1, base_replications + 1),
    upstream_flow_lookup=baseline_upstream_flow_lookup,
    alighting_targets=hourly_alighting_targets,
    travel_offset_lookup=travel_offset_lookup,
    feasibility_backlog=high_demand_feasibility_backlog
)
print("High-Demand budget:", high_demand_ctx["service_budget"],
      "bounds:", high_demand_ctx["min_hourly_trips"], "to", high_demand_ctx["max_hourly_trips"])

High-Demand budget: {'Evening': 44, 'Morning': 43} bounds: 5 to 12


### Reduced-Service hourly bounds

The Reduced-Service GA searches within 4 to 11 trips per hour. This is a scenario-specific bound taken directly from Karol's reduced-service reference plan: 4 is the smallest hourly trip count and 11 is the largest in that reduced schedule, after one trip is removed from every study hour. It is set here explicitly rather than left as an implicit code decision. The fixed period budgets stay at 38 trips for Morning and 39 trips for Evening, and the GA only redistributes those totals across the five hours.

In [43]:
# (new) Reduced-Service context. Demand is unchanged, so it reuses the Baseline
# passenger streams already generated in this notebook. The budget, bounds, and
# reference plan come from Karol's reduced schedule (one trip removed per hour).
reduced_service_budget = reduced_service_plan_in.groupby("Period")["Trips"].sum().to_dict()

# Scenario-specific hourly bounds taken from the reduced reference plan (QA item 9).
reduced_min_hourly_trips = int(reduced_service_plan_in["Trips"].min())
reduced_max_hourly_trips = int(reduced_service_plan_in["Trips"].max())

reduced_reference_plan_lookup = {
    period: tuple(
        reduced_service_plan_in[reduced_service_plan_in["Period"] == period]
        .sort_values("Hour")["Trips"].astype(int).tolist()
    )
    for period in ["Morning", "Evening"]
}
reduced_service_feasibility_backlog = (
    reduced_service_results[reduced_service_results["Replication"].isin(optimization_replications)]
    .groupby("Period")["Final Backlog"].mean().to_dict()
)
reduced_service_reference_backlog_validation = (
    reduced_service_results[reduced_service_results["Replication"].isin(range(1, base_replications + 1))]
    .groupby("Period")["Final Backlog"].mean().to_dict()
)

reduced_service_ctx = make_scenario_context(
    name="Reduced-Service",
    passengers_by_replication=passengers_by_replication,
    seed_table=seed_table,
    service_budget=reduced_service_budget,
    min_hourly_trips=reduced_min_hourly_trips,
    max_hourly_trips=reduced_max_hourly_trips,
    reference_plan_lookup=reduced_reference_plan_lookup,
    search_replication_ids=optimization_replications,
    validation_replication_ids=range(1, base_replications + 1),
    upstream_flow_lookup=baseline_upstream_flow_lookup,
    alighting_targets=hourly_alighting_targets,
    travel_offset_lookup=travel_offset_lookup,
    feasibility_backlog=reduced_service_feasibility_backlog
)
print("Reduced-Service budget:", reduced_service_ctx["service_budget"],
      "bounds:", reduced_min_hourly_trips, "to", reduced_max_hourly_trips)
print("Reduced-Service reference plan:", reduced_reference_plan_lookup)

Reduced-Service budget: {'Evening': 39, 'Morning': 38} bounds: 4 to 11
Reduced-Service reference plan: {'Morning': (4, 11, 11, 7, 5), 'Evening': (7, 11, 10, 7, 4)}


In [44]:
# Run the generalized GA on High-Demand, selecting the final plan with the
# 30-replication final-backlog safeguard.
high_demand_runs, high_demand_selected = run_ga_for_scenario(
    high_demand_ctx, optimizer_seeds=optimizer_seeds, evaluation_budget=evaluation_budget,
    final_backlog_reference=high_demand_reference_backlog_validation
)
high_demand_selected

,Scenario,Period,Technique,Service Plan,Mean Wait,Mean Backlog,Mean Peak Queue,Mean P95 Wait,Mean Period-End Queue,Mean Bus Occupancy,Maximum Bus Occupancy,Passed Final Backlog Safeguard
0,High-Demand,Morning,GA,"(8, 12, 10, 8, 5)",3.476655,2.666667,21.866667,7.519975,6.066667,17.882171,42,True
1,High-Demand,Evening,GA,"(10, 10, 10, 9, 5)",3.500153,1.566667,11.200000,7.781034,4.166667,14.010000,27,True


In [45]:
# Run the generalized GA on Reduced-Service, selecting the final plan with the
# 30-replication final-backlog safeguard.
reduced_service_runs, reduced_service_selected = run_ga_for_scenario(
    reduced_service_ctx, optimizer_seeds=optimizer_seeds, evaluation_budget=evaluation_budget,
    final_backlog_reference=reduced_service_reference_backlog_validation
)
reduced_service_selected

,Scenario,Period,Technique,Service Plan,Mean Wait,Mean Backlog,Mean Peak Queue,Mean P95 Wait,Mean Period-End Queue,Mean Bus Occupancy,Maximum Bus Occupancy,Passed Final Backlog Safeguard
0,Reduced-Service,Morning,GA,"(7, 9, 10, 7, 5)",3.889338,2.100000,18.066667,8.325816,4.466667,17.728947,36,True
1,Reduced-Service,Evening,GA,"(10, 7, 8, 8, 6)",3.934947,1.066667,8.833333,7.926485,2.566667,13.823932,28,True


In [46]:
# (new) Confirm each selected plan respects the scenario safeguards, comparing
# against the reference service on the same validation replications:
#   1. the plan sums to the correct service budget,
#   2. every hour stays within the scenario bounds,
#   3. the plan's mean final backlog does not exceed the reference backlog,
#   4. passenger conservation holds in the validation runs.
def check_scenario_safeguards(ctx, selected_df, runs_df, reference_backlog_validation):
    rows = []
    for _, row in selected_df.iterrows():
        period = row["Period"]
        plan = tuple(int(x) for x in row["Service Plan"])
        rows.append({
            "Scenario": ctx["name"], "Period": period, "Plan": plan,
            "Plan Backlog": round(float(row["Mean Backlog"]), 3),
            "Reference Backlog": round(float(reference_backlog_validation[period]), 3),
            "Budget OK": sum(plan) == ctx["service_budget"][period],
            "Bounds OK": all(ctx["min_hourly_trips"] <= c <= ctx["max_hourly_trips"] for c in plan),
            "Backlog Safeguard OK": row["Mean Backlog"] <= reference_backlog_validation[period] + 1e-9,
            "Passenger Conservation OK": bool(runs_df[runs_df["Period"] == period]["Passenger Conservation"].all())
        })
    return pd.DataFrame(rows)

scenario_safeguards = pd.concat([
    check_scenario_safeguards(high_demand_ctx, high_demand_selected, high_demand_runs, high_demand_reference_backlog_validation),
    check_scenario_safeguards(reduced_service_ctx, reduced_service_selected, reduced_service_runs, reduced_service_reference_backlog_validation)
], ignore_index=True)
all_safeguards_ok = bool(
    scenario_safeguards[["Budget OK", "Bounds OK", "Backlog Safeguard OK", "Passenger Conservation OK"]].values.all()
)
print("All scenario safeguards passed:", all_safeguards_ok)
scenario_safeguards

All scenario safeguards passed: True


,Scenario,Period,Plan,Plan Backlog,Reference Backlog,Budget OK,Bounds OK,Backlog Safeguard OK,Passenger Conservation OK
0,High-Demand,Morning,"(8, 12, 10, 8, 5)",2.667,2.867,True,True,True,True
1,High-Demand,Evening,"(10, 10, 10, 9, 5)",1.567,4.267,True,True,True,True
2,Reduced-Service,Morning,"(7, 9, 10, 7, 5)",2.100,2.167,True,True,True,True
3,Reduced-Service,Evening,"(10, 7, 8, 8, 6)",1.067,3.233,True,True,True,True


In [47]:
# QA item 8: fail-fast. Stop before saving if any selected scenario plan fails
# the required safeguards.
if not all_safeguards_ok:
    raise RuntimeError(
        "Scenario safeguard check failed: a selected plan does not meet budget, "
        "bounds, final-backlog, or passenger-conservation requirements. "
        "Stopping before saving outputs."
    )
print("Scenario safeguard gate passed.")

Scenario safeguard gate passed.


## 10. Selected-Plan Replication-Level Results, Validation, Precision, and Stop-Level

The selected High-Demand and Reduced-Service plans are re-evaluated on all 30 validation replications, keeping the full per-replication rows for the paired comparison and final QA, the per-replication validation flags, the replication precision check, and the stop-level results.

In [48]:
# (new) Full per-replication results and stop-level results for the selected plans.
high_demand_selected_reps, high_demand_stop_level = evaluate_scenario_selected_plans(
    high_demand_ctx, high_demand_selected
)
reduced_service_selected_reps, reduced_service_stop_level = evaluate_scenario_selected_plans(
    reduced_service_ctx, reduced_service_selected
)
selected_plan_replications = pd.concat(
    [high_demand_selected_reps, reduced_service_selected_reps], ignore_index=True
)
selected_plan_stop_level = pd.concat(
    [high_demand_stop_level, reduced_service_stop_level], ignore_index=True
)
print("Selected-plan replication rows:", len(selected_plan_replications))
selected_plan_replications.head()

Selected-plan replication rows: 120


,Scenario,Replication,Seed,Period,Service Plan,Mean Completed Wait,P95 Completed Wait,Peak Queue,Period-End Queue,Final Backlog,Mean Bus Occupancy,Maximum Bus Occupancy,Minimum Bus Occupancy,Minimum Completed Wait,Waiting Time Valid,Bus Occupancy Valid,Passenger Conservation
0,High-Demand,1,479243620,Morning,"(8, 12, 10, 8, 5)",3.340769,7.176688,19,11,5,17.827907,35,6,0.004187,True,True,True
1,High-Demand,2,454024514,Morning,"(8, 12, 10, 8, 5)",3.499999,7.790174,18,8,1,17.920930,38,5,0.013172,True,True,True
2,High-Demand,3,1818751028,Morning,"(8, 12, 10, 8, 5)",3.439254,7.482614,23,4,2,17.897674,41,6,0.002154,True,True,True
3,High-Demand,4,2288458700,Morning,"(8, 12, 10, 8, 5)",3.518985,8.007954,23,7,1,17.920930,36,5,0.001370,True,True,True
4,High-Demand,5,2115331525,Morning,"(8, 12, 10, 8, 5)",3.391855,7.754980,20,8,2,17.897674,36,5,0.004025,True,True,True


In [49]:
# (new) Validation evidence for the selected GA plans (QA item 5): completed
# waiting times non-negative, bus occupancy within 0 to 60, passenger conservation.
validation_rows = []
for scenario_name, reps_df in [
    ("High-Demand", high_demand_selected_reps),
    ("Reduced-Service", reduced_service_selected_reps)
]:
    validation_rows.append({
        "Scenario": scenario_name,
        "Replications Used": int(reps_df["Replication"].nunique()),
        "Non-Negative Waiting Time Passed": bool(reps_df["Waiting Time Valid"].all()),
        "Bus Occupancy In [0, 60] Passed": bool(reps_df["Bus Occupancy Valid"].all()),
        "Passenger Conservation Passed": bool(reps_df["Passenger Conservation"].all()),
        "Minimum Observed Waiting Time": round(float(reps_df["Minimum Completed Wait"].min()), 4),
        "Minimum Observed Bus Occupancy": int(reps_df["Minimum Bus Occupancy"].min()),
        "Maximum Observed Bus Occupancy": int(reps_df["Maximum Bus Occupancy"].max())
    })
selected_validation_summary = pd.DataFrame(validation_rows)
all_validation_passed = bool(
    selected_validation_summary[["Non-Negative Waiting Time Passed",
                                 "Bus Occupancy In [0, 60] Passed",
                                 "Passenger Conservation Passed"]].values.all()
)
print("All selected-plan validation checks passed:", all_validation_passed)
if not all_validation_passed:
    raise RuntimeError(
        "Selected-plan validation failed: a waiting time was negative, a bus "
        "occupancy left 0 to 60, or passenger conservation did not hold. "
        "Stopping before saving outputs."
    )
selected_validation_summary

All selected-plan validation checks passed: True


,Scenario,Replications Used,Non-Negative Waiting Time Passed,"Bus Occupancy In [0, 60] Passed",Passenger Conservation Passed,Minimum Observed Waiting Time,Minimum Observed Bus Occupancy,Maximum Observed Bus Occupancy
0,High-Demand,30,True,True,True,0.0002,1,42
1,Reduced-Service,30,True,True,True,0.0002,1,36


In [50]:
# (new) Replication precision check for the selected GA plans (QA item 6).
# Start at 30 validation replications. If precision is met, keep 30. If not, flag
# that more matched replications are required under the existing 30-to-50 rule.
high_demand_precision, high_demand_precision_ok = compute_precision_check(
    high_demand_selected_reps, "High-Demand"
)
reduced_service_precision, reduced_service_precision_ok = compute_precision_check(
    reduced_service_selected_reps, "Reduced-Service"
)
selected_precision_check = pd.concat(
    [high_demand_precision, reduced_service_precision], ignore_index=True
)
for scenario_name, ok in [("High-Demand", high_demand_precision_ok),
                          ("Reduced-Service", reduced_service_precision_ok)]:
    if ok:
        print(f"{scenario_name}: precision met at {base_replications} replications.")
    else:
        print(f"{scenario_name}: precision NOT met at {base_replications} replications. "
              f"More matched replications required under the {base_replications}-to-{precision_max_replications} rule.")
selected_precision_check

High-Demand: precision met at 30 replications.
Reduced-Service: precision met at 30 replications.


,Scenario,Period,Metric,Replications,Mean,Half-Width,Required Half-Width,Precision Passed
0,High-Demand,Morning,Mean Completed Wait,30,3.4767,0.0307,1.0000,True
1,High-Demand,Morning,Peak Queue,30,21.8667,0.9027,2.1867,True
2,High-Demand,Evening,Mean Completed Wait,30,3.5002,0.0382,1.0000,True
3,High-Demand,Evening,Peak Queue,30,11.2000,0.7558,1.1200,True
4,Reduced-Service,Morning,Mean Completed Wait,30,3.8893,0.0424,1.0000,True
5,Reduced-Service,Morning,Peak Queue,30,18.0667,0.7779,1.8067,True
6,Reduced-Service,Evening,Mean Completed Wait,30,3.9349,0.0578,1.0000,True
7,Reduced-Service,Evening,Peak Queue,30,8.8333,0.4812,1.0000,True


## 11. Paired Reference-versus-GA Comparison (95% Confidence Intervals)

The reference service and the GA plans use the **same scenario demand, the same service budget, and matched passenger streams**, but they represent bus timing differently. The reference service uses the published GTFS scheduled times. The GA candidate uses evenly spaced within-hour buses placed at midpoint headways, with downstream stop times set from the corresponding hour's median GTFS travel offset. Because of this, part of any measured improvement can come from headway regularization, not only from redistributing the hourly number of trips. The paired comparison below matches reference and GA results on Replication, Seed, and Period, computes GA minus Reference per replication, and reports the mean paired difference with a t-based 95% confidence interval. An improvement is treated as statistically supported only when the whole interval is below zero.

In [51]:
# (new) Paired reference-versus-GA differences with 95% confidence intervals.
paired_metrics = ["Mean Completed Wait", "P95 Completed Wait", "Peak Queue", "Final Backlog"]
scenario_paired_ci = pd.concat([
    paired_reference_vs_ga(high_demand_selected_reps, high_demand_results, "High-Demand", paired_metrics),
    paired_reference_vs_ga(reduced_service_selected_reps, reduced_service_results, "Reduced-Service", paired_metrics)
], ignore_index=True)
scenario_paired_ci

,Scenario,Period,Metric,N,Mean GA,Mean Reference,Mean Paired Diff (GA - Reference),CI Lower,CI Upper,Statistically Supported Improvement
0,High-Demand,Morning,Mean Completed Wait,30,3.4767,3.6627,-0.1860,-0.2227,-0.1493,True
1,High-Demand,Morning,P95 Completed Wait,30,7.5200,9.0379,-1.5179,-1.6422,-1.3936,True
2,High-Demand,Morning,Peak Queue,30,21.8667,23.3667,-1.5000,-2.5084,-0.4916,True
3,High-Demand,Morning,Final Backlog,30,2.6667,2.8667,-0.2000,-0.3808,-0.0192,True
4,High-Demand,Evening,Mean Completed Wait,30,3.5002,3.6414,-0.1412,-0.1956,-0.0868,True
5,High-Demand,Evening,P95 Completed Wait,30,7.7810,7.8323,-0.0513,-0.3803,0.2778,False
6,High-Demand,Evening,Peak Queue,30,11.2000,13.9000,-2.7000,-3.3136,-2.0864,True
7,High-Demand,Evening,Final Backlog,30,1.5667,4.2667,-2.7000,-3.4282,-1.9718,True
8,Reduced-Service,Morning,Mean Completed Wait,30,3.8893,4.8501,-0.9608,-1.0393,-0.8822,True
9,Reduced-Service,Morning,P95 Completed Wait,30,8.3258,14.3864,-6.0606,-6.4054,-5.7157,True


In [52]:
# Save all scenario-integration GA outputs under new filenames so nothing from
# Baseline or from Karol's scenario tables is overwritten.
for scen, runs_df, selected_df in [
    ("highdemand", high_demand_runs, high_demand_selected),
    ("reducedservice", reduced_service_runs, reduced_service_selected)
]:
    runs_export = runs_df.copy()
    runs_export["Best Service Plan"] = runs_export["Best Service Plan"].astype(str)
    runs_export.to_csv(tables_dir / f"ga_scenario_{scen}_runs.csv", index=False)
    selected_export = selected_df.copy()
    selected_export["Service Plan"] = selected_export["Service Plan"].astype(str)
    selected_export.to_csv(tables_dir / f"ga_scenario_{scen}_selected_plans.csv", index=False)

selected_reps_export = selected_plan_replications.copy()
selected_reps_export["Service Plan"] = selected_reps_export["Service Plan"].astype(str)
selected_reps_export.to_csv(tables_dir / "ga_scenario_selected_plan_replications.csv", index=False)

selected_plan_stop_level.to_csv(tables_dir / "ga_scenario_stop_level_results.csv", index=False)
selected_validation_summary.to_csv(tables_dir / "ga_scenario_validation_summary.csv", index=False)
selected_precision_check.to_csv(tables_dir / "ga_scenario_precision_check.csv", index=False)
scenario_paired_ci.to_csv(tables_dir / "ga_scenario_paired_reference_vs_ga.csv", index=False)
scenario_safeguards.to_csv(tables_dir / "ga_scenario_safeguards_check.csv", index=False)

# Stop-level averages across replications, for quick reading.
scenario_stop_level_summary = (
    selected_plan_stop_level
    .groupby(["Scenario", "Period", "Stop Sequence"])[
        ["Mean Wait", "P95 Wait", "Peak Queue", "Period-End Queue", "Final Backlog"]
    ].mean().round(3).reset_index()
)
scenario_stop_level_summary.to_csv(tables_dir / "ga_scenario_stop_level_summary.csv", index=False)

print("Saved scenario-integration GA outputs to", tables_dir)
scenario_stop_level_summary

Saved scenario-integration GA outputs to ../output/tables


,Scenario,Period,Stop Sequence,Mean Wait,P95 Wait,Peak Queue,Period-End Queue,Final Backlog
0,High-Demand,Evening,6,3.313,6.509,9.133,1.567,1.567
1,High-Demand,Evening,7,3.463,6.274,3.000,0.000,0.000
2,High-Demand,Evening,8,3.454,7.046,2.533,0.000,0.000
3,High-Demand,Evening,9,3.483,7.123,3.133,0.500,0.000
4,High-Demand,Evening,10,3.798,9.867,11.000,2.100,0.000
5,High-Demand,Morning,6,3.533,7.797,9.667,2.667,2.667
6,High-Demand,Morning,7,3.385,7.092,3.200,0.467,0.000
7,High-Demand,Morning,8,3.548,7.482,2.133,0.367,0.000
8,High-Demand,Morning,9,3.585,7.473,3.500,0.500,0.000
9,High-Demand,Morning,10,3.450,7.433,21.867,2.067,0.000


## 12. Remaining Final Work

This notebook now covers the GA-core Baseline verification and the High-Demand and Reduced-Service GA integration, with final-backlog-safeguarded selection, replication-level outputs, validation evidence, the replication precision check, paired 95% confidence intervals against the reference service, and stop-level results. The remaining Final work is the Combined scenario (demand raised 40 percent and one trip removed per hour at the same time), which is intentionally deferred to the Final paper phase and will reuse this same generalized GA with a Combined context built from the combined passenger streams and the reduced service budget.